# Variable Importance for the Causal Forest

Aim: measure how much of the heterogeneity in the estimated treatment effects each group of customer characteristics accounts for, using the drop-and-retrain importance of Bénard & Josse (2023), *Variable importance for causal forests: breaking down the heterogeneity of treatment effects*.

For a group of variables J, the importance is the share of the variance of the CATE that is lost when J is removed:

It is estimated by retraining the final causal forest without the group's columns on the **same** nuisance residuals (centering always uses all variables), applying the paper's corrective term (equation 4), and subtracting the score of a forest retrained on all columns with a new seed (retraining noise). Values near 0, including small negative ones, mean no importance. Implementation: `src/vimp.py`.

### 1. Imports

In [1]:
import sys
import time
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from econml.dml import CausalForestDML
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyClassifier

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.vimp import vimp_causal_forest

RANDOM_STATE = 1213
DATA_PATH = next(p for p in [PROJECT_ROOT / "data" / "processed" / "retailhero_causal_forest_features.csv",
                             PROJECT_ROOT / "data" / "Processed" / "retailhero_causal_forest_features.csv"] if p.exists())
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"

### 2. Rebuild the training data exactly as in causal_forest.ipynb

In [2]:
df = pd.read_csv(DATA_PATH)
Y = df["target"].astype(int)
Z = df["treatment"].astype(int)
X = df.drop(columns=["target", "treatment", "client_id", "gender_U"], errors="ignore")

strata = Z.astype(str) + "_" + Y.astype(str)
X_train, X_test, Y_train, Y_test, Z_train, Z_test = train_test_split(
    X, Y, Z, test_size=0.3, random_state=RANDOM_STATE, stratify=strata)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (140027, 21), Test: (60012, 21)


### 3. Refit the causal forest with cached residuals

The importance measure reuses the forest's cross-fitted nuisance residuals, which EconML only stores when fitted with `cache_values=True`. The saved model was fitted without them, so the same configuration is refitted here with the same seed. `n_jobs=-1` only speeds up the fit: EconML and scikit-learn draw all random seeds before parallelising, so the model is identical. This is checked against the saved test-set CATEs.

In [3]:
cf = CausalForestDML(
    model_y=RandomForestRegressor(n_estimators=100, min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1),
    model_t=DummyClassifier(strategy="prior", random_state=RANDOM_STATE),
    discrete_treatment=True,
    n_estimators=200,
    min_samples_leaf=20,
    cv=3,
    honest=True,
    inference=True,
    random_state=RANDOM_STATE,
    n_jobs=-1)

cf.fit(Y=Y_train.to_numpy(), T=Z_train.to_numpy(), X=X_train.to_numpy(), cache_values=True)

### 4. Variable groups

In [4]:
groups = {
    "Demographics": ["age", "gender_F", "gender_M"],
    "Spending": ["total_spend", "avg_transaction_spend", "median_transaction_spend", "spend_std"],
    "Frequency/recency": ["num_transactions", "days_since_last_purchase",
                          "avg_days_between_transactions", "customer_activity_span"],
    "Loyalty points": ["regular_points_received", "regular_points_spent", "express_points_received",
                       "express_points_spent", "net_regular_points_change", "net_express_points_change"],
    "Store engagement": ["num_stores_visited", "num_unique_products", "total_quantity",
                         "avg_items_per_transaction"],
}

### 5. Group importance

One forest is retrained per group plus one baseline, on the full training set (n = 140,027), using the out-of-bag CATEs of the original forest as the reference.

In [5]:
vimp_groups = vimp_causal_forest(cf, X_train, groups)
print(f"Baseline retraining bias I_n(0): {vimp_groups.attrs['baseline']:.4f}")
vimp_groups.to_csv(OUTPUT_DIR / "cf_vimp_groups.csv")

Baseline retraining bias I_n(0): 0.5089


Summary

In [6]:
summary = pd.DataFrame({"Importance": vimp_groups,
                        "# Features": [len(groups[g]) for g in vimp_groups.index],
                        "Features": [", ".join(groups[g]) for g in vimp_groups.index]})
display(summary.round(3))

,Importance,# Features,Features
group,,,
Loyalty points,0.334,6,"regular_points_received, regular_points_spent,..."
Spending,0.078,4,"total_spend, avg_transaction_spend, median_tra..."
Frequency/recency,0.073,4,"num_transactions, days_since_last_purchase, av..."
Store engagement,0.063,4,"num_stores_visited, num_unique_products, total..."
Demographics,0.057,3,"age, gender_F, gender_M"


### 6. Drill-down within the most important group

Each column of the top group is scored as its own group. A column that can be reconstructed from the others (for example `net_regular_points_change = regular_points_received + regular_points_spent`), or that is strongly correlated with columns left in the model, scores close to 0 on its own even when the group matters.

In [7]:
top_group = vimp_groups.index[0]
print(f"Drilling down into: {top_group}")
vimp_drill = vimp_causal_forest(cf, X_train, {c: [c] for c in groups[top_group]})
display(vimp_drill.round(4))
vimp_drill.to_csv(OUTPUT_DIR / "cf_vimp_drilldown.csv")

Drilling down into: Loyalty points


group
net_regular_points_change    0.0084
express_points_received      0.0040
regular_points_received      0.0032
express_points_spent         0.0025
net_express_points_change    0.0007
regular_points_spent        -0.0014
Name: importance, dtype: float64